# 04 — Tagging promptbook: dev sample for the two tagging passes

**Purpose.** Develop the Layer-2 prompts on a small stratified sample of
blocks and put every output in front of the researcher. No run larger than
this sample happens until the researcher has read a dev round and approved
it; every wording or format change bumps `tagging.PROMPT_VERSION`.

**What Layer 2 is** (design ratified 2026-08-22): an independent re-measure
of firm-topic relevance. The model reads document-order sentence blocks and
cites indices — the Track-2 discipline applied to Item 1, with the frozen
codebook as a closed answer set plus a direction (opportunity / threat /
neutral). No Layer-1 output enters any prompt; agreement between the layers
is therefore evidence. Intensity is counted from tags downstream
(per filing-year and topic), never asked as a scalar.

**Two passes.** Pass A gates each block against all 111 topic *labels* and
answers with a bare list of topic ids; Pass B confirms with the candidates'
frozen definitions and adds sentence-cited direction records. Two earlier
gate designs were retired on evidence: gating on the 14 Stage-3 themes
(0.1-dev) failed structurally — grab-bag themes made whole subject areas
invisible — and the 1.0 gate's per-topic sentence citations were re-derived
by Pass B anyway while their decoding tripled call time and their length
hit the output cap, silently truncating topic lists (production run
11280399, aborted at ~150 blocks). Grounding stays in the gate's
instruction; citations live in Pass B alone (1.1).

**Inputs.** 03's `blocks.parquet` + 01's sentences; the frozen codebook.

**Outputs.** Dev CallLogs under `tagging/dev/status/`; the printed read.

**Gate.** Parse-clean rate, per-call seconds, and — decisive — the
researcher's reading of every printed block.

## Setup — codebook lists and the dev sample

Eight blocks stratified over eras and firms — the same seed as 0.1-dev, so
this round is read against the identical text.

In [1]:
import hashlib
import os
import sys
import time
from collections import Counter

sys.path.insert(0, "../lib")

import pandas as pd
import torch

from paths import DATA_REPO, HF_CACHE, TOPICS_INTERIM, relevance_run_dir
import tagging as T
import summarize as S

os.environ["HF_HOME"] = str(HF_CACHE)

MODEL = os.environ.get("MODEL", "Qwen/Qwen3.5-4B")
N_DEV = 8
SEED = 1089

ITEM1 = relevance_run_dir("item1")
TAG = relevance_run_dir("tagging")

codebook = pd.read_parquet(TOPICS_INTERIM / "taxonomy" / "codebook_v1.parquet")
LABELS_TEXT, TOPIC_IDS = T.format_topic_labels(codebook)
topic_label = codebook.set_index("topic_id")["label"]
print(f"codebook: {len(TOPIC_IDS)} topics")
print(f"prompt version: {T.PROMPT_VERSION}\n")

blocks = pd.read_parquet(TAG / "blocks.parquet")
meta = pd.read_parquet(
    DATA_REPO / "raw" / "03_sp500_10k_item1" / "sp500_10k_item1.parquet",
    columns=["accessionNo", "conm", "fyear"])
blocks = blocks.merge(meta, on="accessionNo", validate="m:1")
sentences = pd.read_parquet(ITEM1 / "sentences.parquet",
                            columns=["accessionNo", "sent_idx", "sentence"])

blocks["era"] = pd.cut(blocks["fyear"], [0, 2004, 2014, 9999],
                       labels=["1994-2004", "2005-2014", "2015-2025"])
dev = (blocks.groupby("era", observed=True)
       .apply(lambda g: g.sample(min(3, len(g)), random_state=SEED),
              include_groups=False)
       .reset_index(drop=True).head(N_DEV))
sent_by_acc = dict(tuple(
    sentences.loc[sentences["accessionNo"].isin(set(dev["accessionNo"]))]
    .groupby("accessionNo")))
print("dev blocks:")
print(dev[["conm", "fyear", "block_idx", "n_sentences"]]
      .to_string(index=False))

codebook: 111 topics
prompt version: item1-topic-tagging-1.2



dev blocks:
                      conm  fyear  block_idx  n_sentences
AMERICAN ELECTRIC POWER CO   1999         16           40
    KINDRED HEALTHCARE INC   1999          0           40
  MOLSON COORS BEVERAGE CO   2001          7           15
           ROSS STORES INC   2008          2           40
                 ADOBE INC   2008          8           40
                VIACOM INC   2008          1           40
       NEKTAR THERAPEUTICS   2023          3           40
                 AMGEN INC   2019         13           25


## Load the model

Same model, sampling, and per-call seeding discipline as every extraction
stage: seed from content hash, so a re-run reproduces bit-for-bit.

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype="auto",
                                             device_map="cuda")
model.eval()
print(f"loaded {MODEL} in {time.time() - t0:.0f}s")


def seed_for(acc, block_idx, stage):
    torch.manual_seed(int.from_bytes(
        hashlib.sha256(f"{acc}:{block_idx}:{stage}".encode()).digest()[:8],
        "little"))


def block_sentences(row):
    grp = sent_by_acc[row.accessionNo]
    sel = grp.loc[(grp["sent_idx"] >= row.sent_lo)
                  & (grp["sent_idx"] <= row.sent_hi), "sentence"].tolist()
    return sel, row.sent_lo + 1          # display numbering is 1-based

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

loaded Qwen/Qwen3.5-4B in 6s


## Pass A dev — label gate

Every block printed in full with its candidate topics: the reader judges
whether the flagged topics are the ones a person would flag, whether
anything a person would flag is missing (the generosity check), and whether
citations point at the specific matching sentences rather than ranges.

In [3]:
log = S.CallLog(TAG / "dev" / "status" / "promptbook.jsonl",
                "dev", MODEL, prompt_version=T.PROMPT_VERSION)
gate_hits = []
errs_a = Counter()

for row in dev.itertuples():
    sents, start = block_sentences(row)
    user = T.render_gate_user(sents, start, LABELS_TEXT)
    seed_for(row.accessionNo, row.block_idx, "gate")
    t0 = time.time()
    out, hit_cap = S._generate(model, tokenizer, T.SYSTEM, user,
                               T.GATE_MAX_NEW_TOKENS)
    secs = time.time() - t0
    log.write("gate", 0, row.Index, len(sents), user, out, tokenizer, secs)
    topics_hit, errs = T.parse_gate_topics(out, TOPIC_IDS)
    errs_a.update(errs)
    gate_hits.append((row, topics_hit))

    print("=" * 78)
    print(f"{row.conm}  fy{row.fyear}  block {row.block_idx}"
          f"  [{len(sents)} sentences, {secs:.1f}s]"
          f"{'  CAP HIT' if hit_cap else ''}")
    for i, s in enumerate(sents, start):
        print(f"  [{i}] {s[:100]}")
    if not topics_hit:
        print("  -> no topics gated")
    for tid in topics_hit:
        print(f"  -> {tid} {topic_label[tid]}")

AMERICAN ELECTRIC POWER CO  fy1999  block 16  [40 sentences, 2.0s]
  [641] Except as noted herein, AEP's subsidiaries that own or operate generating, transmission and distribu
  [642] For the AEP System, compliance with the Clean Air Act (CAA) is requiring substantial expenditures th
  [643] However, there can be no assurance that all such costs will be recovered.
  [644] See Construction Program -- Construction Expenditures.
  [645] Acid Rain: The Acid Rain Program (Title IV) of the Clean Air Act Amendments of 1990 (CAAA) created a
  [646] There are two phases of SO(2) control under the Acid Rain Program.
  [647] Phase I, effective January 1, 1995, required SO(2) emission reductions from certain units that emitt
  [648] Phase II, which affects all fossil fuel-fired steam generating units with capacity greater than 25 m
  [649] If a unit emitted SO(2) in 1985 at a rate in excess of 1.2 pounds per million Btu heat input, the Ph
  [650] In addition to regulating SO(2) emissions, Title IV

KINDRED HEALTHCARE INC  fy1999  block 0  [40 sentences, 0.9s]
  [1] Vencor, Inc. ("Vencor" or the "Company") provides long-term healthcare services primarily through th
  [2] At December 31, 1999, the Company's health services division operated 295 nursing centers (38,573 li
  [3] The Company's hospital division operated 56 hospitals (4,931 licensed beds) in 23 states and an inst
  [4] During 1999, the Company operated its Vencare ancillary services business which provided respiratory
  [5] As a result of significant declines in the demand for ancillary services caused by the Balanced Budg
  [6] Vencare's rehabilitation, speech and occupational therapies were integrated into the Company's nursi
  [7] Vencare's institutional pharmacy business was assigned to the hospital division.
  [8] Vencare's respiratory therapy and other ancillary businesses have been discontinued.
  [9] The Company and substantially all of its subsidiaries filed voluntary petitions for protection under
  [10] The 

MOLSON COORS BEVERAGE CO  fy2001  block 7  [15 sentences, 0.7s]
  [281] ; Philip Morris, Inc., through its subsidiary Miller Brewing Company; and Coors Brewing Company.
  [282] We compete most directly with Anheuser and Miller, the dominant companies in the U.S. industry.
  [283] According to Beer Marketer's Insights estimates, we are the nation's third-largest brewer, selling a
  [284] This compares to Anheuser's 48.6% share and Miller's 19.6% share.
  [285] Our beer shipments to wholesalers decreased 1.2% in 2001, primarily because fiscal 2000 was 53 weeks
  [286] On a comparable- calendar basis, our 2001 beer shipments were down 0.1% from a year earlier because 
  [287] By comparison, Anheuser's domestic shipments increased 1.2% in 2001 and Miller's declined 2.4%.
  [288] More than 85% of our unit volume was in the premium and above-premium price categories, the highest 
  [289] We continue to face competitive disadvantages related to economies of scale.
  [290] Besides lower transp

ROSS STORES INC  fy2008  block 2  [40 sentences, 0.8s]
  [81] Consistent with the other aspects of our business strategy, we strive to keep operating costs as low
  [82] Among the factors which have enabled us to keep operating costs low are:
  [83] Labor costs that generally are lower than full-price department and specialty stores due to
  [84] (i) a store design that creates a self-service retail format and
  [85] (ii) the utilization of labor saving technologies.
  [86] Economies of scale with respect to general and administrative costs as a result of centralized merch
  [87] Flexible store layout criteria which facilitates conversion of existing buildings to our format.
  [88] In fiscal 2008, we continued to invest in new information systems and technology to provide a platfo
  [89] Recent initiatives include the following:
  [90] We completed development and began the rollout of demand forecasting software and related process ch
  [91] We expect this initiative to drive gradual i

ADOBE INC  fy2008  block 8  [40 sentences, 1.0s]
  [321] We also intend to release new versions of our Adobe AIR client to broaden its capabilities and we in
  [322] We also recently announced and intend to release Adobe Flash Catalyst, a new professional design too
  [323] As part of the Open Screen Project, we have announced that we are removing the OEM licensing fees as
  [324] We intend to build upon the OEM relationships we have to monetize the downloads of our client techno
  [325] By leveraging our developer engineering, marketing and evangelism capabilities, we intend to impleme
  [326] To support our Adobe Flash Platform initiatives, we intend to continue our marketing efforts through
  [327] We will also work closely with partners to implement Adobe AIR client distribution relationships and
  [328] We intend to continue our focus in improving integration of our Platform technologies with our Creat
  [329] Finally, we intend to deliver new product capabilities and Web-based se

VIACOM INC  fy2008  block 1  [40 sentences, 4.0s]
  [41] For example, MTV targets teen and young adult demographics, Nickelodeon targets kids and their famil
  [42] The advertising industry modified the way it measures ratings in 2008 by moving to commercial rating
  [43] Commercial ratings are measured on a “C3” basis, which counts the number of viewers that watch the c
  [44] In 2008, the majority of our guaranteed deals were sold on the C3 metric.
  [45] We regularly evaluate the structure, content and volume of our advertising spots, and throughout 200
  [46] In 2008, domestic and global economic conditions worsened significantly, which had a rapid, negative
  [47] Current economic conditions have adversely affected our advertising revenues and such effect could c
  [48] Our advertising revenues may also fluctuate due to the ratings of our channels (including the timing
  [49] Some of our program services experienced ratings declines in 2008, which, coupled with economic cond
  [50

NEKTAR THERAPEUTICS  fy2023  block 3  [40 sentences, 1.1s]
  [121] The FDA may deny approval if applicable regulatory criteria are not satisfied or may require additio
  [122] Even if such data are submitted, the FDA may ultimately decide that the NDA or BLA does not satisfy 
  [123] Additionally, the approved labeling may narrowly limit the conditions of use of the product, includi
  [124] Further, as a condition of approval, the FDA may impose post-market surveillance, or Phase 4, studie
  [125] Product approvals, once obtained, may be withdrawn if compliance with regulatory standards is not ma
  [126] The FDA may require additional post-marketing clinical testing and pharmacovigilance programs to mon
  [127] After approval, there are ongoing reporting obligations concerning adverse reactions associated with
  [128] Each manufacturing establishment producing the active pharmaceutical ingredient and finished drug pr
  [129] Such inspections are also held periodically after commerciali

AMGEN INC  fy2019  block 13  [25 sentences, 0.8s]
  [521] From 2010 to 2012, Ms. Patton was Vice President, Law.
  [522] Prior to joining the Company, Ms. Patton served as Senior Vice President, General Counsel and Secret
  [523] Mr. David A. Piacquad, age 63, became Senior Vice President, Business Development in 2014.
  [524] Mr. Piacquad joined the Company in 2010 and served as Vice President, Strategy and Corporate Develop
  [525] Prior to joining the Company, from 2009 to 2010, Mr. Piacquad was Principal of David A. Piacquad Con
  [526] From 2006 to 2009, Mr. Piacquad served as Senior Vice President, Business Development and Licensing,
  [527] Prior to Schering-Plough, Mr. Piacquad served in a series of leadership roles in finance and busines
  [528] Dr. David M. Reese, age 57, became Executive Vice President, R&D, in 2018.
  [529] Dr. Reese joined the Company in 2005 and has held leadership roles in development, medical sciences 
  [530] Dr. Reese was Senior Vice President, Transl

## Pass B dev — confirmation + direction

One call per block carrying all of that block's candidate topics with their
frozen definitions. Printed with the cited sentence resolved, so
misgrounded tags are visible immediately.

In [4]:
errs_b = Counter()
n_b_calls = 0
all_tags = []

for row, topics_hit in gate_hits:
    if not topics_hit:
        continue
    sents, start = block_sentences(row)
    candidates = sorted(topics_hit)
    entries, cand_ids = T.format_topic_entries(codebook, candidates)
    user = T.render_confirm_user(sents, start, entries)
    seed_for(row.accessionNo, row.block_idx, "confirm")
    t0 = time.time()
    out, hit_cap = S._generate(model, tokenizer, T.SYSTEM, user,
                               T.CONFIRM_MAX_NEW_TOKENS)
    n_b_calls += 1
    log.write("confirm", 0, row.Index, len(sents), user, out, tokenizer,
              time.time() - t0)
    tags, errs = T.parse_topic_tags(out, cand_ids, start, len(sents))
    errs_b.update(errs)
    all_tags.extend(tags)

    print("=" * 78)
    print(f"{row.conm}  fy{row.fyear}  block {row.block_idx}  "
          f"candidates {candidates}{'  CAP HIT' if hit_cap else ''}")
    if not tags:
        print("  (no tags confirmed)")
    for t in tags:
        print(f"  [{t['sentence']}] {sents[t['sentence'] - start][:80]}")
        print(f"     -> {t['topic']} {topic_label[t['topic']]} "
              f"({t['direction']})")

AMERICAN ELECTRIC POWER CO  fy1999  block 16  candidates [78]
  [641] Except as noted herein, AEP's subsidiaries that own or operate generating, trans
     -> 78 EPA air and water regulations (neutral)
  [642] For the AEP System, compliance with the Clean Air Act (CAA) is requiring substan
     -> 78 EPA air and water regulations (threat)
  [643] However, there can be no assurance that all such costs will be recovered.
     -> 78 EPA air and water regulations (neutral)
  [645] Acid Rain: The Acid Rain Program (Title IV) of the Clean Air Act Amendments of 1
     -> 78 EPA air and water regulations (neutral)
  [646] There are two phases of SO(2) control under the Acid Rain Program.
     -> 78 EPA air and water regulations (neutral)
  [647] Phase I, effective January 1, 1995, required SO(2) emission reductions from cert
     -> 78 EPA air and water regulations (neutral)
  [648] Phase II, which affects all fossil fuel-fired steam generating units with capaci
     -> 78 EPA air and water re

VIACOM INC  fy2008  block 1  candidates [47, 48, 52, 54, 59, 61, 65, 68, 69, 92, 93, 94, 119, 121]
  (no tags confirmed)


NEKTAR THERAPEUTICS  fy2023  block 3  candidates [83]
  (no tags confirmed)


## Machine summary

Counts the reader's eye can miss: parse failures, out-of-range citations,
direction balance. High `unknown_topic` or `index_out_of_range` means the
prompt, not the corpus, needs work.

In [5]:
n_a = len(dev)
print(f"Pass A: {n_a} calls, errors: {dict(errs_a) or 'none'}")
print(f"Pass B: {n_b_calls} calls, errors: {dict(errs_b) or 'none'}")
print(f"confirmed tags: {len(all_tags)}")
if all_tags:
    print("directions:", dict(Counter(t['direction'] for t in all_tags)))
gated = sum(len(r) for _, r in gate_hits)
print(f"gate candidates: {gated} across {sum(1 for _, r in gate_hits if r)} blocks")
print(f"\ncalls logged: {len(log.records)} -> {log.path}")

Pass A: 8 calls, errors: none
Pass B: 3 calls, errors: none
confirmed tags: 39
directions: {'neutral': 31, 'threat': 8}
gate candidates: 16 across 3 blocks

calls logged: 11 -> /scratch2/acroh/projects/llm_method/data/interim/relevance/tagging/dev/status/promptbook.jsonl


## Next steps (nothing below runs without the researcher's check)

1. Researcher reads this round; prompt wording iterates here until the read
   is clean; each change bumps `tagging.PROMPT_VERSION`.
2. Freeze (drop `-dev`).
3. `05` — no-gate audit (random blocks, Pass B over all 111 topics,
   measures Pass A's miss rate) + human gold on a random tag sample.
4. `06` — production sbatch over all 168,333 blocks (Pass A) and gated
   blocks (Pass B); aggregation to (firm, fyear, topic) cells with
   intensity = tagged-sentence counts; the hearing join stays INV-25 (most
   recent 10-K filed before the hearing date).